# 05 — Evidencia de Optimización y Análisis FinOps

**Proyecto:** Plataforma Big Data para Analítica Omnicanal de Ventas Retail  
**Equipo:** Ballerini · Torres · Vargas · Vásquez  
**Fase:** 2 — Implementación y Optimización (evidencia complementaria)

---

Este notebook consolida las métricas de rendimiento antes/después de las optimizaciones aplicadas al pipeline  
y el análisis de costos cloud (FinOps) de la arquitectura implementada.

| # | Optimización | Técnica |
|---|---|---|
| 1 | Caching de la capa Silver | `df.cache()` antes de los jobs Gold |
| 2 | Particionamiento de Silver | `.partitionBy("anio", "mes")` |
| 3 | Tuning de Shuffle Partitions | `spark.sql.shuffle.partitions = 8` |

> **Declaración de IA:** Este notebook fue asistido por Claude (Anthropic) para estructura y comentarios.  
> Los tiempos de ejecución son medidos con `time.time()` en ejecución real sobre Google Colab + GCS.


## 0. Configuración del Entorno

In [1]:
# ── Instalación de dependencias ────────────────────────────────────────────────
!pip uninstall -y dataproc-spark-connect opentelemetry-api importlib-metadata pyspark delta-spark > /dev/null
!pip install -q importlib-metadata==8.0.0 pyspark==3.4.1 delta-spark==2.4.0


In [2]:
# ── Configuración GCS y Spark ──────────────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
print("Autenticación GCP exitosa.")


Autenticación GCP exitosa.


In [3]:
import pyspark, os, time
import pandas as pd

pyspark_jars_dir = os.path.join(pyspark.__path__[0], "jars")
!wget -q https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar -P {pyspark_jars_dir}
print("Conector GCS descargado.")


Conector GCS descargado.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, countDistinct
from delta import configure_spark_with_delta_pip

# ── SparkSession con tuning de shuffle partitions ────────────────────────────
builder = (
    SparkSession.builder
    .appName("Evidencia_Optimizacion_Retail")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # ✅ OPTIMIZACIÓN 3: reducir shuffle partitions de 200 → 8
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)
print("Shuffle partitions configuradas:", spark.conf.get("spark.sql.shuffle.partitions"))


Spark: 3.4.1
Shuffle partitions configuradas: 8


In [5]:
NOMBRE_BUCKET = "data-lake-retail"
RUTA_BASE     = f"gs://{NOMBRE_BUCKET}"
RUTA_SILVER   = f"{RUTA_BASE}/silver"
RUTA_GOLD     = f"{RUTA_BASE}/gold"

# Leer la capa Silver (necesaria para los benchmarks)
df_silver = spark.read.format("delta").load(f"{RUTA_SILVER}/venta_tiendas_delta")
total_registros = df_silver.count()
print(f"Registros en Silver: {total_registros:,}")
df_silver.printSchema()


Registros en Silver: 2,249,970
root
 |-- id_canal: integer (nullable = true)
 |-- numero_transaccion: long (nullable = true)
 |-- numero_pos: integer (nullable = true)
 |-- numero_boleta: long (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: integer (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: long (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- venta: double (nullable = true)
 |-- costo: double (nullable = true)
 |-- fecha_transaccion_guion: string (nullable = true)
 |-- fecha_timestamp: timestamp (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- fecha_venta_texto: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- margen: double (nullable = true)
 |-- margen_porcentaje: double (nullable = true)



---
## 1. Optimización 1: Caching de la Capa Silver

**Hipótesis:** Al construir los 4 data marts Gold, el motor Spark reejecutará el plan de transformación  
completo desde Bronze para cada job si el DataFrame no está cacheado.  
Cacherlo en memoria debería reducir significativamente el tiempo de los jobs subsiguientes.


In [6]:
# ════════════════════════════════════════════════════════════════════════════
# BENCHMARK 1A — Consulta agregada SIN cache
# ════════════════════════════════════════════════════════════════════════════
print("BASELINE — Sin cache:")
print("─" * 50)

inicio = time.time()
resultado_sin_cache = (
    df_silver
    .groupBy("anio", "mes")
    .agg(
        spark_sum("venta").alias("venta_total"),
        spark_sum("margen").alias("margen_total"),
        countDistinct("numero_boleta").alias("boletas")
    )
)
resultado_sin_cache.count()  # Materializa la acción
tiempo_baseline = time.time() - inicio

print(f"Consulta: ventas mensuales (groupBy anio, mes)")
print(f"Tiempo SIN cache: {tiempo_baseline:.2f} segundos")


BASELINE — Sin cache:
──────────────────────────────────────────────────
Consulta: ventas mensuales (groupBy anio, mes)
Tiempo SIN cache: 487.06 segundos


In [7]:
# ════════════════════════════════════════════════════════════════════════════
# OPTIMIZACIÓN: Persistir Silver en memoria
# ════════════════════════════════════════════════════════════════════════════
print("Cacheando Silver en memoria...")
df_silver_cached = df_silver.cache()
# Trigger de materialización (warm-up del cache)
df_silver_cached.count()
print("Cache completado.")


Cacheando Silver en memoria...
Cache completado.


In [8]:
# ════════════════════════════════════════════════════════════════════════════
# BENCHMARK 1B — Misma consulta CON cache
# ════════════════════════════════════════════════════════════════════════════
print("OPTIMIZADO — Con cache:")
print("─" * 50)

inicio = time.time()
resultado_con_cache = (
    df_silver_cached
    .groupBy("anio", "mes")
    .agg(
        spark_sum("venta").alias("venta_total"),
        spark_sum("margen").alias("margen_total"),
        countDistinct("numero_boleta").alias("boletas")
    )
)
resultado_con_cache.count()
tiempo_cache = time.time() - inicio

mejora_pct = ((tiempo_baseline - tiempo_cache) / tiempo_baseline) * 100

print(f"Consulta: ventas mensuales (groupBy anio, mes)")
print(f"Tiempo CON cache: {tiempo_cache:.2f} segundos")
print(f"Mejora: −{mejora_pct:.1f}%")


OPTIMIZADO — Con cache:
──────────────────────────────────────────────────
Consulta: ventas mensuales (groupBy anio, mes)
Tiempo CON cache: 1.50 segundos
Mejora: −99.7%


In [9]:
# ════════════════════════════════════════════════════════════════════════════
# BENCHMARK 1C — Consulta con filtro temporal (simula patron Gold por tienda)
# ════════════════════════════════════════════════════════════════════════════

# Sin cache (leer de nuevo sin persistencia)
df_silver_cold = spark.read.format("delta").load(f"{RUTA_SILVER}/venta_tiendas_delta")
inicio = time.time()
df_silver_cold.filter((col("anio") == 2016) & (col("mes") == 3))     .groupBy("cod_tienda_facturacion")     .agg(spark_sum("venta").alias("venta_total"))     .count()
t_filtro_sin = time.time() - inicio

# Con cache (misma consulta sobre df_silver_cached)
inicio = time.time()
df_silver_cached.filter((col("anio") == 2016) & (col("mes") == 3))     .groupBy("cod_tienda_facturacion")     .agg(spark_sum("venta").alias("venta_total"))     .count()
t_filtro_con = time.time() - inicio

mejora_filtro = ((t_filtro_sin - t_filtro_con) / t_filtro_sin) * 100

print("Consulta: ventas por tienda, marzo 2016")
print(f"Sin cache: {t_filtro_sin:.2f} s")
print(f"Con cache: {t_filtro_con:.2f} s")
print(f"Mejora:   −{mejora_filtro:.1f}%")


Consulta: ventas por tienda, marzo 2016
Sin cache: 0.77 s
Con cache: 0.53 s
Mejora:   −31.3%


In [10]:
# ════════════════════════════════════════════════════════════════════════════
# TABLA RESUMEN — Optimización 1: Caching
# ════════════════════════════════════════════════════════════════════════════
tabla_cache = spark.createDataFrame([
    ("Ventas por año/mes (groupBy)",      "Sin cache", round(tiempo_baseline, 2)),
    ("Ventas por año/mes (groupBy)",      "Con cache", round(tiempo_cache, 2)),
    ("Ventas por tienda (filtro mes)",    "Sin cache", round(t_filtro_sin, 2)),
    ("Ventas por tienda (filtro mes)",    "Con cache", round(t_filtro_con, 2)),
], ["consulta", "escenario", "tiempo_segundos"])

print("=" * 65)
print("TABLA 1 — Impacto del Caching sobre la Capa Silver")
print("=" * 65)
tabla_cache.show(truncate=False)

# Guardar evidencia en Gold
tabla_cache.write.format("delta").mode("overwrite")     .save(f"{RUTA_GOLD}/evidencia_cache")
print("✅ Evidencia guardada en Gold.")


TABLA 1 — Impacto del Caching sobre la Capa Silver
+------------------------------+---------+---------------+
|consulta                      |escenario|tiempo_segundos|
+------------------------------+---------+---------------+
|Ventas por año/mes (groupBy)  |Sin cache|487.06         |
|Ventas por año/mes (groupBy)  |Con cache|1.5            |
|Ventas por tienda (filtro mes)|Sin cache|0.77           |
|Ventas por tienda (filtro mes)|Con cache|0.53           |
+------------------------------+---------+---------------+

✅ Evidencia guardada en Gold.


---
## 2. Optimización 2: Particionamiento por Año/Mes

**Hipótesis:** Al escribir Silver particionado por `(anio, mes)`, Spark aplicará **partition pruning**  
en consultas con predicados temporales, leyendo solo los archivos de la partición relevante  
en lugar de escanear el dataset completo.


In [11]:
# ════════════════════════════════════════════════════════════════════════════
# Verificar estructura de particionamiento de Silver
# ════════════════════════════════════════════════════════════════════════════
from pyspark.sql.functions import countDistinct

# Contar particiones lógicas disponibles
particiones = df_silver.select("anio", "mes").distinct().orderBy("anio", "mes")
n_particiones = particiones.count()

print(f"Particiones lógicas (anio, mes) en Silver: {n_particiones}")
print()
print("Períodos disponibles:")
particiones.show(30, truncate=False)


Particiones lógicas (anio, mes) en Silver: 99

Períodos disponibles:
+----+---+
|anio|mes|
+----+---+
|2015|1  |
|2015|2  |
|2015|3  |
|2015|4  |
|2015|5  |
|2015|7  |
|2015|8  |
|2015|11 |
|2016|1  |
|2016|2  |
|2016|3  |
|2016|5  |
|2016|7  |
|2016|9  |
|2016|10 |
|2016|11 |
|2016|12 |
|2017|1  |
|2017|3  |
|2017|6  |
|2017|7  |
|2017|8  |
|2017|9  |
|2017|10 |
|2017|11 |
|2017|12 |
|2018|2  |
|2018|4  |
|2018|5  |
|2018|6  |
+----+---+
only showing top 30 rows



In [12]:
# ════════════════════════════════════════════════════════════════════════════
# BENCHMARK 2 — Partition Pruning: lectura completa vs. partición específica
# ════════════════════════════════════════════════════════════════════════════

# Escenario A: Leer Silver SIN particionamiento (simulado leyendo y filtrando en memoria)
inicio = time.time()
conteo_completo = df_silver.count()
t_scan_completo = time.time() - inicio

# Escenario B: Filtrar aprovechando particionamiento (Spark aplica pruning automático)
inicio = time.time()
conteo_particion = df_silver.filter(
    (col("anio") == 2016) & (col("mes") == 3)
).count()
t_scan_particion = time.time() - inicio

pct_datos_leidos = (1 / n_particiones) * 100
reduccion = ((t_scan_completo - t_scan_particion) / t_scan_completo) * 100

print("=" * 65)
print("TABLA 2 — Impacto del Particionamiento (Partition Pruning)")
print("=" * 65)
print(f"Total particiones disponibles: {n_particiones}")
print(f"Total registros: {conteo_completo:,}")
print(f"Registros en partición anio=2016/mes=3: {conteo_particion:,}")
print()
print(f"Tiempo scan completo (todos los datos): {t_scan_completo:.2f} s")
print(f"Tiempo con partition pruning (1/{n_particiones}): {t_scan_particion:.2f} s")
print(f"Datos leídos con pruning: ~{pct_datos_leidos:.1f}% del total")
print(f"Reducción de tiempo: −{reduccion:.1f}%")


TABLA 2 — Impacto del Particionamiento (Partition Pruning)
Total particiones disponibles: 99
Total registros: 2,249,970
Registros en partición anio=2016/mes=3: 35,391

Tiempo scan completo (todos los datos): 0.24 s
Tiempo con partition pruning (1/99): 0.42 s
Datos leídos con pruning: ~1.0% del total
Reducción de tiempo: −-76.5%


In [13]:
# Guardar tabla de evidencia de particionamiento
tabla_particionamiento = spark.createDataFrame([
    ("Scan completo (sin partition pruning)", conteo_completo,     round(t_scan_completo, 2),   100.0),
    ("Partition pruning (anio=2016, mes=3)", conteo_particion,     round(t_scan_particion, 2),  round(pct_datos_leidos, 1)),
], ["escenario", "registros_leidos", "tiempo_segundos", "pct_datos_escaneados"])

tabla_particionamiento.show(truncate=False)

tabla_particionamiento.write.format("delta").mode("overwrite")     .save(f"{RUTA_GOLD}/evidencia_particionamiento")
print("✅ Evidencia guardada en Gold.")


+-------------------------------------+----------------+---------------+--------------------+
|escenario                            |registros_leidos|tiempo_segundos|pct_datos_escaneados|
+-------------------------------------+----------------+---------------+--------------------+
|Scan completo (sin partition pruning)|2249970         |0.24           |100.0               |
|Partition pruning (anio=2016, mes=3) |35391           |0.42           |1.0                 |
+-------------------------------------+----------------+---------------+--------------------+

✅ Evidencia guardada en Gold.


---
## 3. Optimización 3: Tuning de Shuffle Partitions

**Hipótesis:** El valor por defecto de `spark.sql.shuffle.partitions = 200` genera 200 tareas pequeñas  
para operaciones de shuffle (groupBy, join). En un entorno de 4 vCPU (Google Colab),  
reducirlo a 8 elimina el overhead de gestionar 192 tareas vacías o casi vacías.


In [14]:
# ════════════════════════════════════════════════════════════════════════════
# BENCHMARK 3 — Shuffle partitions: 200 vs 8
# ════════════════════════════════════════════════════════════════════════════

# Escenario A: Con 200 shuffle partitions (default)
spark.conf.set("spark.sql.shuffle.partitions", "200")
inicio = time.time()
df_silver.groupBy("cod_tienda_facturacion", "anio", "mes")     .agg(spark_sum("venta").alias("venta_total"))     .count()
t_200 = time.time() - inicio
print(f"Tiempo con 200 shuffle partitions: {t_200:.2f} s")

# Escenario B: Con 8 shuffle partitions (optimizado)
spark.conf.set("spark.sql.shuffle.partitions", "8")
inicio = time.time()
df_silver.groupBy("cod_tienda_facturacion", "anio", "mes")     .agg(spark_sum("venta").alias("venta_total"))     .count()
t_8 = time.time() - inicio
print(f"Tiempo con 8 shuffle partitions:   {t_8:.2f} s")
print(f"Mejora: −{((t_200-t_8)/t_200)*100:.1f}%")


Tiempo con 200 shuffle partitions: 1.48 s
Tiempo con 8 shuffle partitions:   0.87 s
Mejora: −41.6%


In [15]:
tabla_shuffle = spark.createDataFrame([
    ("groupBy tienda/anio/mes",  200, round(t_200, 2), "Default — genera overhead de tareas vacías"),
    ("groupBy tienda/anio/mes",  8,   round(t_8, 2),   "Tuneado — mapea con los 4 vCPU disponibles"),
], ["consulta", "shuffle_partitions", "tiempo_segundos", "nota"])

print("=" * 65)
print("TABLA 3 — Impacto del Tuning de Shuffle Partitions")
print("=" * 65)
tabla_shuffle.show(truncate=False)


TABLA 3 — Impacto del Tuning de Shuffle Partitions
+-----------------------+------------------+---------------+------------------------------------------+
|consulta               |shuffle_partitions|tiempo_segundos|nota                                      |
+-----------------------+------------------+---------------+------------------------------------------+
|groupBy tienda/anio/mes|200               |1.48           |Default — genera overhead de tareas vacías|
|groupBy tienda/anio/mes|8                 |0.87           |Tuneado — mapea con los 4 vCPU disponibles|
+-----------------------+------------------+---------------+------------------------------------------+



---
## 4. Análisis de Costos — FinOps

La arquitectura fue diseñada siguiendo el principio de **clúster efímero**:  
el cluster de Dataproc se crea solo durante la ventana de procesamiento y se destruye al finalizar.  
Esto evita el costo de compute en las 23+ horas restantes del día.


In [16]:
# ════════════════════════════════════════════════════════════════════════════
# MODELO DE COSTOS — Pipeline Batch Diario
# ════════════════════════════════════════════════════════════════════════════
import pandas as pd

costos = {
    "Componente": [
        "Dataproc n2-standard-4 (master + 2 workers)",
        "Google Cloud Storage (50 GB datos + logs)",
        "BigQuery — Gold + Governance (on-demand)",
        "Cloud Composer — Small environment",
        "[Alternativa] Cloud Workflows",
        "TOTAL (con Composer)",
        "TOTAL (con Workflows)"
    ],
    "Tipo": [
        "Compute efímero (0.25 h/día × 30 días)",
        "Almacenamiento persistente",
        "Queries analíticas (~100 GB/mes)",
        "Orquestación managed",
        "Orquestación serverless",
        "—", "—"
    ],
    "Precio unitario": [
        "USD 1.20 /h",
        "USD 0.02 /GB/mes",
        "USD 0.005 /GB",
        "USD 31.00 /mes fijo",
        "USD 0.01 /1000 pasos",
        "—", "—"
    ],
    "Costo/mes (USD)": [
        9.00, 1.00, 0.50, 31.00, 0.01, 41.50, 10.51
    ]
}

df_costos = pd.DataFrame(costos)
print("=" * 90)
print("TABLA 4 — Estimación de Costos Mensuales")
print("=" * 90)
print(df_costos.to_string(index=False))
print()
print("Supuestos:")
print("  - Cluster Dataproc: 1 master n2-standard-4 + 2 workers n2-standard-4")
print("  - Tiempo de ejecución: 15 min/día × 30 días = 7.5 h/mes")
print("  - Precio referencial GCP us-central1 (verificar calculadora oficial)")
print("  - GCS Standard: USD 0.02/GB/mes (menos de 1% del costo total)")
print("  - BigQuery: primeros 1 TB/mes de queries gratuitos en free tier")


TABLA 4 — Estimación de Costos Mensuales
                                 Componente                                   Tipo      Precio unitario  Costo/mes (USD)
Dataproc n2-standard-4 (master + 2 workers) Compute efímero (0.25 h/día × 30 días)          USD 1.20 /h             9.00
  Google Cloud Storage (50 GB datos + logs)             Almacenamiento persistente     USD 0.02 /GB/mes             1.00
   BigQuery — Gold + Governance (on-demand)       Queries analíticas (~100 GB/mes)        USD 0.005 /GB             0.50
         Cloud Composer — Small environment                   Orquestación managed  USD 31.00 /mes fijo            31.00
              [Alternativa] Cloud Workflows                Orquestación serverless USD 0.01 /1000 pasos             0.01
                       TOTAL (con Composer)                                      —                    —            41.50
                      TOTAL (con Workflows)                                      —                    —         

In [17]:
# ════════════════════════════════════════════════════════════════════════════
# COMPARATIVA: Clúster efímero vs clúster permanente
# ════════════════════════════════════════════════════════════════════════════
horas_mes         = 24 * 30          # = 720 h/mes si el cluster fuera permanente
horas_uso_real    = 0.25 * 30        # = 7.5 h/mes con cluster efímero
precio_hora       = 1.20             # USD/h (referencial)

costo_permanente  = horas_mes * precio_hora
costo_efimero     = horas_uso_real * precio_hora
ahorro            = costo_permanente - costo_efimero
ahorro_pct        = (ahorro / costo_permanente) * 100

print("=" * 65)
print("COMPARATIVA FinOps: Clúster Permanente vs Efímero")
print("=" * 65)
print(f"Horas/mes cluster permanente:  {horas_mes} h → USD {costo_permanente:.0f}/mes")
print(f"Horas/mes cluster efímero:     {horas_uso_real} h  → USD {costo_efimero:.2f}/mes")
print(f"Ahorro mensual:                USD {ahorro:.2f} ({ahorro_pct:.1f}%)")
print()
print("Conclusión: el patrón efímero reduce el costo de compute en un")
print(f"  {ahorro_pct:.0f}%, de USD {costo_permanente:.0f}/mes a USD {costo_efimero:.2f}/mes.")


COMPARATIVA FinOps: Clúster Permanente vs Efímero
Horas/mes cluster permanente:  720 h → USD 864/mes
Horas/mes cluster efímero:     7.5 h  → USD 9.00/mes
Ahorro mensual:                USD 855.00 (99.0%)

Conclusión: el patrón efímero reduce el costo de compute en un
  99%, de USD 864/mes a USD 9.00/mes.


In [18]:
# ════════════════════════════════════════════════════════════════════════════
# RESUMEN EJECUTIVO — Todas las optimizaciones
# ════════════════════════════════════════════════════════════════════════════

resumen = spark.createDataFrame([
    ("Caching Silver (job agregado)", 
     f"{tiempo_baseline:.1f}s", f"{tiempo_cache:.1f}s", 
     f"−{((tiempo_baseline-tiempo_cache)/tiempo_baseline)*100:.0f}%",
     "df_silver.cache() antes de jobs Gold"),
    ("Caching Silver (filtro tienda/mes)",
     f"{t_filtro_sin:.1f}s", f"{t_filtro_con:.1f}s",
     f"−{((t_filtro_sin-t_filtro_con)/t_filtro_sin)*100:.0f}%",
     "Reutilización del mismo cache"),
    ("Partition pruning (1 mes de N)",
     f"{t_scan_completo:.1f}s", f"{t_scan_particion:.1f}s",
     f"~−{((t_scan_completo-t_scan_particion)/t_scan_completo)*100:.0f}%",
     ".partitionBy('anio','mes') en Silver"),
    ("Shuffle partitions (200 → 8)",
     f"{t_200:.1f}s", f"{t_8:.1f}s",
     f"−{((t_200-t_8)/t_200)*100:.0f}%",
     "Tuneado a vCPUs disponibles"),
    ("Clúster efímero vs permanente",
     f"USD {costo_permanente:.0f}/mes", f"USD {costo_efimero:.2f}/mes",
     f"−{ahorro_pct:.0f}%",
     "Cluster se destruye post-pipeline"),
], ["optimizacion", "antes", "despues", "mejora", "tecnica"])

print("=" * 100)
print("RESUMEN EJECUTIVO — Evidencia de Optimización (Fase 2)")
print("=" * 100)
resumen.show(truncate=False)

# Guardar resumen en Gold
resumen.write.format("delta").mode("overwrite")     .save(f"{RUTA_GOLD}/evidencia_resumen_optimizacion")
print("✅ Resumen de optimización guardado en Gold.")
print()
print("Todos los datasets de evidencia disponibles en:")
print(f"  gs://data-lake-retail/gold/evidencia_cache")
print(f"  gs://data-lake-retail/gold/evidencia_particionamiento")
print(f"  gs://data-lake-retail/gold/evidencia_resumen_optimizacion")


RESUMEN EJECUTIVO — Evidencia de Optimización (Fase 2)
+----------------------------------+-----------+------------+------+------------------------------------+
|optimizacion                      |antes      |despues     |mejora|tecnica                             |
+----------------------------------+-----------+------------+------+------------------------------------+
|Caching Silver (job agregado)     |487.1s     |1.5s        |−100% |df_silver.cache() antes de jobs Gold|
|Caching Silver (filtro tienda/mes)|0.8s       |0.5s        |−31%  |Reutilización del mismo cache       |
|Partition pruning (1 mes de N)    |0.2s       |0.4s        |~−-77%|.partitionBy('anio','mes') en Silver|
|Shuffle partitions (200 → 8)      |1.5s       |0.9s        |−42%  |Tuneado a vCPUs disponibles         |
|Clúster efímero vs permanente     |USD 864/mes|USD 9.00/mes|−99%  |Cluster se destruye post-pipeline   |
+----------------------------------+-----------+------------+------+-----------------------------

In [21]:
spark.stop()
print("SparkSession cerrada.")

SparkSession cerrada.
